In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from pgmpy.causal_discovery import HillClimbSearch, ExpertKnowledge
from pgmpy.structure_score import BIC, BICGauss
import matplotlib.pyplot as plt
import pygraphviz
import graphviz
import shutil
import importlib
from IPython.display import Image, display
from sklearn.metrics import f1_score
from collections import Counter
from tqdm import tqdm
from pgmpy.base import DAG
import pickle
import re
from pandas.api.types import CategoricalDtype

c:\Users\Vincent\anaconda3\envs\eurostar-thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option("display.max_columns", None)
# pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.0f}".format)

# Overall structure discovery

In [ ]:
restriction_groups = {'cleanliness': ['Toilets'],
                      'comfort':     ['Air Conditioning', 'Heating', 'Windows'],
                      'wifi':        ['WiFi']}
pm_groups = {'cleanliness': ['toilet'],
             'comfort':     ['climate', 'interior', 'reliability'],
             'services':    ['comms', 'wifi']}
cm_groups = {'cleanliness': ['toilet', 'cleaning'],
             'comfort':     ['climate', 'interior'],
             'wifi':        ['wifi']}

In [4]:
main_block = pd.read_parquet("../data/analysis/aggregated/main_block_aggregated.parquet")
main_block = main_block.reset_index(drop=True)

In [5]:
metadata_columns = ['trainset', 'equipment_type', 'train_service', 'route_type', 'origin_theoretical_time', 'destination_theoretical_time']
main_block = main_block.drop(columns=metadata_columns)
name_mapping = {
    'question_recommendation_score': 'Recommendation',
    'question_overall_rating': 'Satisfaction',
    'question_ticket_price_satisfa': 'Price_Sat',
    'question_overall_satisfaction_overall_service_from_eurostar_staff': 'Staff_Sat',
    'question_overall_satisfaction_journey_punctuality': 'Punctuality_Sat',
    'question_overall_satisfaction_booking_experience': 'Booking_Sat',
    'question_overall_satisfaction_information_provided_to_you_before_travelling': 'Info_Sat',
    'question_overall_satisfaction_experience_at_departure_station': 'Departure_Sat',
    'question_overall_satisfaction_comfort_onboard_the_train': 'Comfort_Sat',
    'question_overall_satisfaction_cleanliness_onboard_the_train': 'Cleanliness_Sat',
    'question_overall_satisfaction_wifi_onboard_the_train': 'WiFi_Sat'
}

main_block.rename(columns=name_mapping, inplace=True)

In [6]:
sat_cols = main_block.columns[main_block.columns.str.endswith('_Sat')].tolist()
main_block_questions = sat_cols + ['Satisfaction', 'Recommendation'] 
print(f"Sample size before dropping missing values: {len(main_block)}")
main_block = main_block.dropna(subset=main_block_questions)
print(f"Sample size after dropping missing values: {len(main_block)}")

Sample size before dropping missing values: 29832
Sample size after dropping missing values: 27177


In [7]:
main_block.head()

,nps,Recommendation,Staff_Sat,Punctuality_Sat,Booking_Sat,Info_Sat,Departure_Sat,Comfort_Sat,Cleanliness_Sat,Satisfaction,Price_Sat,question_strategic_pillar_ass_travelling_by_eurostar_is_an_environmentally_sustainable_option,question_strategic_pillar_ass_eurostar_manages_its_food_and_drink_offering_at_station_lounges_and_onboard_in_an_environmentally_sustainable_way,WiFi_Sat,service_id,restriction_open_Toilets,restriction_open_Air Conditioning,restriction_open_Refrigeration,restriction_open_Windows,restriction_open_WiFi,restriction_open_Heating,total restrictions,restriction_days_Toilets,restriction_days_Air Conditioning,restriction_days_Refrigeration,restriction_days_Windows,restriction_days_WiFi,restriction_days_Heating,longest restriction,pm_days_since_catering,pm_days_since_toilet,pm_days_since_climate,pm_days_since_interior,pm_days_since_reliability,pm_days_since_comms,pm_days_since_wifi,average days since last exams,pm_has_prior_catering,pm_has_prior_toilet,pm_has_prior_climate,pm_has_prior_interior,pm_has_prior_reliability,pm_has_prior_comms,pm_has_prior_wifi,cm_open_climate,cm_open_wifi,cm_open_interior,cm_open_catering,cm_open_toilet,cm_open_cleaning,total open faults,clean_score_routine,clean_hours_since_routine,clean_score_deep,clean_days_since_deep,last_clean_score,hours_since_last_clean,clean_has_prior_routine,clean_has_prior_deep,lda_hours_since,n_trainsets,Exceeded Rotation Time,early_journey_delay_minute,arrival_delay_minute,compensation_liability_evouchers,compensation_liability_cash
0,0,6,4,4,3,3,3,3,3,3,3,4,3,4,9937_20250101,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28,NaN,10,19,0,0,0,0,1,0,1,11,0,38,8,17,3,77,NaN,NaN,NaN,NaN,NaN,NaN,0,0,12,1,00:00:00,0,8,0.00,0.00
1,-100,5,4,1,4,4,2,2,2,2,2,4,4,2,9351_20250101,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27,NaN,127,77,0,0,0,0,1,0,1,4,0,22,9,11,4,50,NaN,NaN,NaN,NaN,NaN,NaN,0,0,14,1,00:00:00,20,37,0.00,0.00
2,-100,5,4,1,4,4,2,2,2,2,2,4,4,2,9351_20250101,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27,NaN,127,77,0,0,0,0,1,0,1,4,0,22,9,11,4,50,NaN,NaN,NaN,NaN,NaN,NaN,0,0,14,1,00:00:00,20,37,0.00,0.00
3,0,7,4,2,4,4,3,3,3,3,2,4,3,4,9451_20250101,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,NaN,89,50,0,0,0,0,1,0,1,5,0,6,15,6,2,34,NaN,NaN,NaN,NaN,NaN,NaN,0,0,14,1,00:00:00,20,20,0.00,0.00
4,71,9,4,4,5,5,4,5,5,5,3,4,3,3,9142_20250101,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63,45,95,58,202,2648,NaN,518,1,1,1,1,1,1,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,11,2,00:03:43,-4,-1,0.00,0.00


In [ ]:
out = pd.DataFrame(index=main_block.index)

def slug(s): # 'Air Conditioning' -> 'air_conditioning'
    return re.sub(r'\W+', '_', s.strip().lower()).strip('_')

restriction_cats = list(dict.fromkeys(c for cats in restriction_groups.values() for c in cats))
pm_cats          = list(dict.fromkeys(c for cats in pm_groups.values()          for c in cats))
cm_cats          = list(dict.fromkeys(c for cats in cm_groups.values()          for c in cats))

def never_as_max2(values, present_mask):
    """Continuous 'time since' encoding: keep real durations where it happened,
       set 'never' to 2 x (max observed duration) so it's an extreme value"""
    values   = pd.to_numeric(values, errors='coerce')
    sentinel = values[present_mask].max() * 2 if present_mask.any() else 0.0
    return values.where(present_mask, sentinel).fillna(sentinel).astype(float)

# 1) RESTRICTIONS -> count of active restrictions + avg duration (none -> 0)
for c in restriction_cats:
    s = slug(c)
    out[f'restr_open_{s}'] = main_block[f'restriction_open_{c}']
    out[f'restr_days_{s}'] = main_block[f'restriction_days_{c}'].fillna(0)

# 2) PREVENTIVE MAINTENANCE -> days since
for c in pm_cats:
    s   = slug(c)
    has = main_block[f'pm_has_prior_{c}'] == 1
    out[f'pm_days_{s}'] = never_as_max2(main_block[f'pm_days_since_{c}'], has)

# 3) CORRECTIVE MAINTENANCE -> per-category open faults
for c in cm_cats:
    out[f'cm_open_{slug(c)}'] = main_block[f'cm_open_{c}']

# 3b) Operational outcomes
out['early_journey_delay_minute'] = main_block['early_journey_delay_minute']
out['arrival_delay_minute']       = main_block['arrival_delay_minute']
out['exceeded_rotation_minutes'] = main_block['Exceeded Rotation Time'].apply(
    lambda t: t.hour * 60 + t.minute + t.second / 60 if pd.notna(t) else 0)

# 3c) Cleaning
has = main_block['last_clean_score'].notna()
out['last_clean_score']       = main_block['last_clean_score'].fillna(0).astype(float)
out['hours_since_last_clean'] = never_as_max2(main_block['hours_since_last_clean'], has)

# 4) Survey questions + LDA (never -> 2x max)
out[main_block_questions] = main_block[main_block_questions]
lda = main_block['lda_hours_since']
out['hours_since_lda'] = never_as_max2(lda, lda.notna())

const_cols = [c for c in out.columns if out[c].nunique(dropna=False) <= 1]
print("Dropping constant columns:", const_cols)
out.drop(columns=const_cols, inplace=True)

Dropping constant columns: []


In [9]:
print(len(out))
out.head()

27177


,restr_open_toilets,restr_days_toilets,restr_open_air_conditioning,restr_days_air_conditioning,restr_open_heating,restr_days_heating,restr_open_windows,restr_days_windows,restr_open_wifi,restr_days_wifi,pm_days_toilet,pm_days_climate,pm_days_interior,pm_days_reliability,pm_days_comms,pm_days_wifi,cm_open_toilet,cm_open_cleaning,cm_open_climate,cm_open_interior,cm_open_wifi,early_journey_delay_minute,arrival_delay_minute,exceeded_rotation_minutes,last_clean_score,hours_since_last_clean,Staff_Sat,Punctuality_Sat,Booking_Sat,Info_Sat,Departure_Sat,Comfort_Sat,Cleanliness_Sat,Price_Sat,WiFi_Sat,Satisfaction,Recommendation,hours_since_lda
0,0,0,0,0,0,0,0,0,0,0,8288,4754,1398,28,6730,10,17,3,11,38,0,0,8,0,0,3409,4,4,3,3,3,3,3,3,4,3,6,12
1,0,0,0,0,0,0,0,0,0,0,8288,4754,1398,27,6730,127,11,4,4,22,0,20,37,0,0,3409,4,1,4,4,2,2,2,2,2,2,5,14
2,0,0,0,0,0,0,0,0,0,0,8288,4754,1398,27,6730,127,11,4,4,22,0,20,37,0,0,3409,4,1,4,4,2,2,2,2,2,2,5,14
3,0,0,0,0,0,0,0,0,0,0,8288,4754,1398,12,6730,89,6,2,5,6,0,20,20,0,0,3409,4,2,4,4,3,3,3,2,4,3,7,14
4,0,0,0,0,0,0,0,0,0,0,45,95,58,202,2648,574,0,0,0,0,0,-4,-1,4,0,3409,4,4,5,5,4,5,5,3,3,5,9,11


In [ ]:
aspect_cols = ['Staff_Sat', 'Punctuality_Sat', 'Booking_Sat', 'Info_Sat', 'Departure_Sat',
               'Comfort_Sat', 'Cleanliness_Sat', 'Price_Sat', 'WiFi_Sat']
sinks   = ['Satisfaction', 'Recommendation']
survey  = aspect_cols + sinks
operational_cols = [c for c in out.columns if c not in survey]

# map each operational column to its analysis category
analysis_cat = lambda k: 'wifi' if k == 'services' else k
col_category = {}
for cat, subs in restriction_groups.items():
    for c in subs:
        col_category[f'restr_open_{slug(c)}'] = analysis_cat(cat)
        col_category[f'restr_days_{slug(c)}'] = analysis_cat(cat)
for cat, subs in pm_groups.items():
    for c in subs:
        col_category[f'pm_days_{slug(c)}'] = analysis_cat(cat)   # no more pm_recency_*
for c in ['last_clean_score', 'hours_since_last_clean']:
    col_category[c] = 'cleanliness'
for cat, subs in cm_groups.items():
    for c in subs:
        col_category[f'cm_open_{slug(c)}'] = analysis_cat(cat)
for c in ['early_journey_delay_minute', 'arrival_delay_minute', 'exceeded_rotation_minutes']:
    col_category[c] = 'punctuality'
for c in ['compensation_evouchers', 'compensation_cash']:
    col_category[c] = 'compensation'
col_category['hours_since_lda'] = 'cleanliness'


uncategorised = [c for c in operational_cols if c not in col_category]
print("Operational nodes:", operational_cols)
print("Uncategorised (free to any aspect):", uncategorised)   # -> hours_since_lda

# allowed operational -> aspect links
category_allowed_aspects = {
    'cleanliness':  ['Cleanliness_Sat', 'Comfort_Sat'],          # cleanliness <-> comfort ambiguous
    'comfort':      ['Comfort_Sat', 'Cleanliness_Sat'],          # comfort <-> cleanliness ambiguous
    'wifi':         ['WiFi_Sat'],                                 # wifi & comms -> WiFi only
    'punctuality':  ['Punctuality_Sat', 'Comfort_Sat', 'Price_Sat'],  # comfort
    'compensation': ['Price_Sat'],
}

temporal_order = [operational_cols, aspect_cols, ['Satisfaction'], ['Recommendation']]
required_edges = [("Satisfaction", "Recommendation")]

# (1) operational KPIs never relate to each other
forbidden_op_op = [(a, b) for a in operational_cols for b in operational_cols if a != b]
# (2) survey aspects never cause each other
forbidden_aspect_aspect = [(a, b) for a in aspect_cols for b in aspect_cols if a != b]
# (3) sinks emit nothing except the required Satisfaction -> Recommendation
forbidden_sinks = [(s, v) for s in sinks for v in out.columns
                   if v != s and (s, v) != ("Satisfaction", "Recommendation")]
# (4) cross-category bans: each op may only point to its category's allowed aspects
forbidden_category = [(c, a) for c, cat in col_category.items() if c in out.columns
                      for a in aspect_cols if a not in category_allowed_aspects[cat]]
# recency shares its "never" level with the score
coupled = [('routine_recency', 'clean_score_routine'), ('deep_recency', 'clean_score_deep')]
forbidden_coupled = [e for a, b in coupled for e in ((a, b), (b, a))
                     if a in out.columns and b in out.columns]

forbidden_edges = list(set(forbidden_op_op + forbidden_aspect_aspect + forbidden_sinks
                           + forbidden_category + forbidden_coupled))

expert_knowledge = ExpertKnowledge(temporal_order=temporal_order,
                                   required_edges=required_edges,
                                   forbidden_edges=forbidden_edges)
print(f"Tiers:     {[len(t) for t in temporal_order]}")
print(f"Required:  {len(required_edges)}")
print(f"Forbidden: {len(forbidden_edges)}")

Operational nodes: ['restr_open_toilets', 'restr_days_toilets', 'restr_open_air_conditioning', 'restr_days_air_conditioning', 'restr_open_heating', 'restr_days_heating', 'restr_open_windows', 'restr_days_windows', 'restr_open_wifi', 'restr_days_wifi', 'pm_days_toilet', 'pm_days_climate', 'pm_days_interior', 'pm_days_reliability', 'pm_days_comms', 'pm_days_wifi', 'cm_open_toilet', 'cm_open_cleaning', 'cm_open_climate', 'cm_open_interior', 'cm_open_wifi', 'early_journey_delay_minute', 'arrival_delay_minute', 'exceeded_rotation_minutes', 'last_clean_score', 'hours_since_last_clean', 'hours_since_lda']
Uncategorised (free to any aspect): []
Tiers:     [27, 9, 1, 1]
Required:  1
Forbidden: 1038


In [31]:
hc = HillClimbSearch(scoring_method="bic-g", max_indegree=10, return_type="dag", expert_knowledge=expert_knowledge)
hc.fit(out)

  0%|          | 21/1000000 [00:14<196:39:42,  1.41it/s]


KeyboardInterrupt: 

In [ ]:
overall_structure = hc.causal_graph_
overall_graph = overall_structure.to_graphviz()

In [ ]:
for node in overall_graph.nodes():
    original = node.get_name()
    short = name_mapping.get(original, original)
    node.attr.update({
        "label": short,
        "shape": "box",
        "style": "rounded",
        "fontname": "Helvetica",
        "fontsize": "12",
        "fixedsize": "false",
        "width": "0",
        "height": "0",
        "margin": "0.2,0.1",
        "penwidth": "1.0",
    })

overall_graph.graph_attr.update({
    "rankdir": "LR",
    "size": "3,4",
    "dpi": "1200",
    #"nodesep": "0.5",
    #"ranksep": "0.8",
    "splines": "polyline",
    "bgcolor": "white",
    "pack": True,
})

overall_graph.edge_attr.update({
    "arrowsize": "0.3",
    "penwidth": "0.8",
    "color": "#303030",
})

overall_graph.layout(prog="dot")
# display(overall_graph)
overall_graph.draw("initial_overall_graph_10_indegree.jpg")

## Max in-degree nodes tuning

In [ ]:
scorer = BICGauss(out)
results = {}

for k in range(3, 11):
    hc = HillClimbSearch(
        scoring_method="bic-g",
        max_indegree=k,
        max_iter=int(1e4),
        return_type="dag",
        expert_knowledge=expert_knowledge,
        show_progress=False,
    )
    hc.fit(out)
    model = hc.causal_graph_
    results[k] = {"model": model,
                  "bic_gauss": scorer.score(model),
                  "n_edges": len(model.edges())}
    print(f"max_indegree={k}:  BIC(Gauss)={results[k]['bic_gauss']:>16,.1f}   edges={results[k]['n_edges']}")

# pgmpy's BICGauss is HIGHER-is-better
best_k = max(results, key=lambda k: results[k]["bic_gauss"])
best_model = results[best_k]["model"]
print(f"\nBest max_indegree = {best_k}  (highest Gaussian BIC = lowest textbook BIC)")

max_indegree=3:  BIC(Gauss)=    -3,262,526.2   edges=18
max_indegree=4:  BIC(Gauss)=    -3,261,351.7   edges=20
max_indegree=5:  BIC(Gauss)=    -3,260,549.4   edges=22
max_indegree=6:  BIC(Gauss)=    -3,260,220.3   edges=24
max_indegree=7:  BIC(Gauss)=    -3,260,056.6   edges=26
max_indegree=8:  BIC(Gauss)=    -3,259,876.9   edges=28
max_indegree=9:  BIC(Gauss)=    -3,259,816.2   edges=30
max_indegree=10:  BIC(Gauss)=    -3,259,804.3   edges=32

Best max_indegree = 10  (highest Gaussian BIC = lowest textbook BIC)


## Bootstrapping

In [ ]:
B = 750
THRESHOLD = 0.5 
rng = np.random.default_rng(42)

edge_counts = Counter()
n_ok = 0

for b in tqdm(range(B)):
    sample = out.sample(n=len(out), replace=True, random_state=int(rng.integers(1e9)))
    # safety: a resample of a sparse column can go constant
    consts = [c for c in sample.columns if sample[c].nunique() <= 1]
    try:
        hc = HillClimbSearch(scoring_method="bic-g", max_indegree=10, return_type="dag",
                             expert_knowledge=expert_knowledge, show_progress=False)
        hc.fit(sample.drop(columns=consts) if consts else sample)
        edge_counts.update(list(hc.causal_graph_.edges()))
        n_ok += 1
    except Exception as e:
        print(f"bootstrap {b} skipped: {type(e).__name__}: {e}")

# edges that appear in > 50% of structures
edge_freq = {e: c / n_ok for e, c in edge_counts.items()}
stable_edges = sorted([(e, f) for e, f in edge_freq.items() if f > THRESHOLD],
                      key=lambda x: -x[1])

print(f"\n{n_ok}/{B} fits OK | {len(stable_edges)} stable edges (>{THRESHOLD:.0%}):\n")
for (u, v), f in stable_edges:
    print(f"  {f:5.0%}  {u} -> {v}")

100%|██████████| 750/750 [3:58:58<00:00, 19.12s/it]  


750/750 fits OK | 31 stable edges (>50%):

   100%  pm_days_comms -> WiFi_Sat
   100%  pm_days_wifi -> WiFi_Sat
   100%  early_journey_delay_minute -> Punctuality_Sat
   100%  early_journey_delay_minute -> Price_Sat
   100%  arrival_delay_minute -> Punctuality_Sat
   100%  arrival_delay_minute -> Comfort_Sat
   100%  arrival_delay_minute -> Price_Sat
   100%  arrival_delay_minute -> Satisfaction
   100%  Staff_Sat -> Satisfaction
   100%  Staff_Sat -> Recommendation
   100%  Punctuality_Sat -> Satisfaction
   100%  Booking_Sat -> Satisfaction
   100%  Info_Sat -> Recommendation
   100%  Info_Sat -> Satisfaction
   100%  Departure_Sat -> Satisfaction
   100%  Departure_Sat -> Recommendation
   100%  Comfort_Sat -> Satisfaction
   100%  Comfort_Sat -> Recommendation
   100%  Price_Sat -> Satisfaction
   100%  Price_Sat -> Recommendation
   100%  WiFi_Sat -> Satisfaction
   100%  Satisfaction -> Recommendation
   100%  early_journey_delay_minute -> Comfort_Sat
    98%  exceeded_rotation_

In [ ]:
used_nodes = {n for e, _ in stable_edges for n in e}
viz = DAG()
viz.add_nodes_from(used_nodes)
viz.add_edges_from([e for e, _ in stable_edges])

g = viz.to_graphviz()

for node in g.nodes():
    node.attr.update({
        "shape": "box", "style": "rounded", "fontname": "Helvetica",
        "fontsize": "20",
        "fixedsize": "false", "width": "0", "height": "0",
        "margin": "0.35,0.22",
        "penwidth": "1.0",
    })

for e in g.edges():
    u, v = e
    f = edge_freq[(u, v)]
    e.attr.update({
        "label": f" {f:.2f}",
        "penwidth": str(0.3 + 1.0 * f),   # was 0.6+3.0*f -> thinner (50%->0.8, 100%->1.3)
        "arrowsize": "0.4",               # was 0.6 -> smaller arrowheads
        "color": "#444444",
        "fontsize": "5", "fontcolor":"#404040", "fontname": "Helvetica",
        "labelfloat": "false",

    })

g.graph_attr.update({
    "rankdir": "LR", "dpi": "1200", "size": 12,
    "nodesep": "0.5", "ranksep": "1.2",
    "splines": "polyline", "overlap": "false", "bgcolor": "white",
    # NOTE: removed "size": 3  — it capped the canvas at 3 inches and shrank
    # everything. Drop it (or set e.g. "size": "12") so bigger nodes actually render big.
})

g.layout(prog="dot")
g.draw("bootstrapped_overall_structure.jpg")       # uncomment to save

In [ ]:
bootstrapped_overall = {"edge_counts": dict(edge_counts), "edge_freq": edge_freq,
        "stable_edges": stable_edges, "n_ok": n_ok, "B": B}
with open("bootstrapped_overall_results.pkl", "wb") as f:
    pickle.dump(bootstrapped_overall, f)

# Main block questions discovery

In [ ]:
main_questions = pd.read_parquet("../data/analysis/individual/main_block_individual.parquet").reset_index(drop=True)
main_questions = main_questions.loc[:, main_questions.columns.str.startswith('question_')]
main_questions.dropna(inplace=True)
main_questions.drop(columns=['question_qid61_nps_group', 'question_strategic_pillar_ass_travelling_by_eurostar_is_an_environmentally_sustainable_option','question_strategic_pillar_ass_eurostar_manages_its_food_and_drink_offering_at_station_lounges_and_onboard_in_an_environmentally_sustainable_way'], inplace=True)
main_questions.rename(columns=name_mapping, inplace=True)
print(len(main_questions))
main_questions.head()

42005


,Recommendation,Staff_Sat,Punctuality_Sat,Booking_Sat,Info_Sat,Departure_Sat,Comfort_Sat,Cleanliness_Sat,Satisfaction,Price_Sat,WiFi_Sat
0,2,3,3,1,1,3,1,1,1,1,2
1,8,4,4,4,3,2,4,4,5,4,4
2,9,5,5,5,5,5,5,5,4,5,5
4,5,3,1,3,4,2,3,2,3,4,2
6,5,5,1,4,4,3,4,3,3,2,3


In [ ]:
main_questions = main_questions.astype("category")
print(main_questions.dtypes)
print({c: main_questions[c].cat.categories.tolist() for c in main_questions.columns})

Recommendation     category
Staff_Sat          category
Punctuality_Sat    category
Booking_Sat        category
Info_Sat           category
Departure_Sat      category
Comfort_Sat        category
Cleanliness_Sat    category
Satisfaction       category
Price_Sat          category
WiFi_Sat           category
dtype: object
{'Recommendation': [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0], 'Staff_Sat': [1, 2, 3, 4, 5], 'Punctuality_Sat': [1, 2, 3, 4, 5], 'Booking_Sat': [1, 2, 3, 4, 5], 'Info_Sat': [1, 2, 3, 4, 5], 'Departure_Sat': [1, 2, 3, 4, 5], 'Comfort_Sat': [1, 2, 3, 4, 5], 'Cleanliness_Sat': [1, 2, 3, 4, 5], 'Satisfaction': [1, 2, 3, 4, 5], 'Price_Sat': [1, 2, 3, 4, 5], 'WiFi_Sat': [1, 2, 3, 4, 5]}


In [172]:
temporal_order_survey = [
    ['Booking_Sat'],
    ['Info_Sat'], 
    ['Departure_Sat'],
    ['Punctuality_Sat'],
    ['Comfort_Sat', 'Cleanliness_Sat', 'WiFi_Sat', 'Staff_Sat'],
    ['Price_Sat'],
    ['Satisfaction'],
    ['Recommendation'],
]

forbidden_survey = [(s, v) for s in sinks for v in main_questions.columns
                   if v != s and (s, v) != ("Satisfaction", "Recommendation")]

expert_knowledge_survey = ExpertKnowledge(
    temporal_order=temporal_order_survey,
    required_edges=required_edges,
    forbidden_edges=forbidden_survey,
)
print(f"Tiers:     {[len(t) for t in temporal_order_survey]}")
print(f"Required:  {len(required_edges)}")
print(f"Forbidden: {len(forbidden_survey)}")   # -> 381

Tiers:     [1, 1, 1, 1, 4, 1, 1, 1]
Required:  1
Forbidden: 19


In [ ]:
hc_survey = HillClimbSearch(scoring_method='bds', max_indegree=10, return_type='dag', expert_knowledge=expert_knowledge_survey)
hc_survey.fit(main_questions)

  0%|          | 20/1000000 [00:00<5:31:26, 50.28it/s]


,scoring_method,'bds'
,start_dag,None
,tabu_length,100
,max_indegree,10
,expert_knowledge,Expert Knowle...ch space edges
,return_type,'dag'
,epsilon,0.0001
,max_iter,1000000
,show_progress,True


In [ ]:
survey_structure = hc_survey.causal_graph_
survey_graph = survey_structure.to_graphviz()
survey_graph.graph_attr.update(rankdir='TB', dpi='1200')
survey_graph.layout(prog='dot')
display(survey_graph)
survey_graph.draw('initial_survey_structure.jpg')

In [181]:
survey_structure.edges()

OutEdgeView([('Staff_Sat', 'WiFi_Sat'), ('Punctuality_Sat', 'Satisfaction'), ('Punctuality_Sat', 'Cleanliness_Sat'), ('Punctuality_Sat', 'Comfort_Sat'), ('Booking_Sat', 'Info_Sat'), ('Booking_Sat', 'Price_Sat'), ('Booking_Sat', 'Departure_Sat'), ('Info_Sat', 'Departure_Sat'), ('Info_Sat', 'Punctuality_Sat'), ('Info_Sat', 'Cleanliness_Sat'), ('Departure_Sat', 'Punctuality_Sat'), ('Departure_Sat', 'Satisfaction'), ('Departure_Sat', 'Staff_Sat'), ('Comfort_Sat', 'Staff_Sat'), ('Comfort_Sat', 'Satisfaction'), ('Comfort_Sat', 'Price_Sat'), ('Comfort_Sat', 'WiFi_Sat'), ('Cleanliness_Sat', 'Comfort_Sat'), ('Cleanliness_Sat', 'Staff_Sat'), ('Satisfaction', 'Recommendation'), ('Price_Sat', 'Recommendation')])

In [182]:
scorer_tuning = BIC(main_questions)
results_tuning = {}

for k in range(3, 11):
    hc_tuning = HillClimbSearch(
        scoring_method="bds",
        max_indegree=k,
        return_type="dag",
        expert_knowledge=expert_knowledge_survey,
        show_progress=False,
    )
    hc_tuning.fit(main_questions)
    model_tuning = hc_tuning.causal_graph_
    results_tuning[k] = {"model": model_tuning,
                  "bic": scorer_tuning.score(model_tuning),
                  "n_edges": len(model_tuning.edges())}
    print(f"max_indegree={k}:  BIC={results_tuning[k]['bic']:>16,.1f}   edges={results_tuning[k]['n_edges']}")

best_k_tuning = max(results_tuning, key=lambda k: results_tuning[k]["bic"])
best_model_tuning = results_tuning[best_k_tuning]["model"]
print(f"\nBest max_indegree = {best_k_tuning}")

max_indegree=3:  BIC=      -541,154.0   edges=21
max_indegree=4:  BIC=      -541,154.0   edges=21
max_indegree=5:  BIC=      -541,154.0   edges=21
max_indegree=6:  BIC=      -541,154.0   edges=21
max_indegree=7:  BIC=      -541,154.0   edges=21
max_indegree=8:  BIC=      -541,154.0   edges=21
max_indegree=9:  BIC=      -541,154.0   edges=21
max_indegree=10:  BIC=      -541,154.0   edges=21

Best max_indegree = 3


In [ ]:
B = 1000
THRESHOLD = 0.5 
rng = np.random.default_rng(42)

edge_counts = Counter()
n_ok = 0

for b in tqdm(range(B)):
    sample = main_questions.sample(n=len(main_questions), replace=True, random_state=int(rng.integers(1e9)))
    # safety: a resample of a sparse column can go constant
    consts = [c for c in sample.columns if sample[c].nunique() <= 1]
    try:
        hc_bootstrap = HillClimbSearch(scoring_method="bds", max_indegree=10, return_type="dag",
                             expert_knowledge=expert_knowledge_survey, show_progress=False)
        hc_bootstrap.fit(sample.drop(columns=consts) if consts else sample)
        edge_counts.update(list(hc_bootstrap.causal_graph_.edges()))
        n_ok += 1
    except Exception as e:
        print(f"bootstrap {b} skipped: {type(e).__name__}: {e}")

# edges that appear in > 50% of structures
edge_freq = {e: c / n_ok for e, c in edge_counts.items()}
stable_edges = sorted([(e, f) for e, f in edge_freq.items() if f > THRESHOLD],
                      key=lambda x: -x[1])

print(f"\n{n_ok}/{B} fits OK | {len(stable_edges)} stable edges (>{THRESHOLD:.0%}):\n")
for (u, v), f in stable_edges:
    print(f"  {f:5.0%}  {u} -> {v}")

consensus = DAG()
consensus.add_nodes_from(main_questions.columns)
consensus.add_edges_from([e for e, _ in stable_edges])
print("\nconsensus acyclic:", nx.is_directed_acyclic_graph(consensus))

100%|██████████| 1000/1000 [08:11<00:00,  2.03it/s]


1000/1000 fits OK | 21 stable edges (>50%):

   100%  Punctuality_Sat -> Satisfaction
   100%  Booking_Sat -> Info_Sat
   100%  Booking_Sat -> Price_Sat
   100%  Booking_Sat -> Departure_Sat
   100%  Info_Sat -> Departure_Sat
   100%  Info_Sat -> Punctuality_Sat
   100%  Departure_Sat -> Punctuality_Sat
   100%  Comfort_Sat -> Satisfaction
   100%  Comfort_Sat -> Price_Sat
   100%  Comfort_Sat -> WiFi_Sat
   100%  Satisfaction -> Recommendation
   100%  Price_Sat -> Recommendation
   100%  Departure_Sat -> Satisfaction
    97%  Staff_Sat -> WiFi_Sat
    92%  Cleanliness_Sat -> Staff_Sat
    91%  Cleanliness_Sat -> Comfort_Sat
    90%  Punctuality_Sat -> Cleanliness_Sat
    89%  Departure_Sat -> Staff_Sat
    87%  Info_Sat -> Cleanliness_Sat
    80%  Comfort_Sat -> Staff_Sat
    75%  Punctuality_Sat -> Comfort_Sat

consensus acyclic: True


In [48]:
bootstrapped_survey = {"edge_counts": dict(edge_counts), "edge_freq": edge_freq, "stable_edges": stable_edges, "n_ok": n_ok, "B": B}
with open("bootstrapped_survey_results.pkl", "wb") as f:
    pickle.dump(bootstrapped_survey, f)

In [ ]:
used_nodes = {n for e, _ in stable_edges for n in e}
viz = DAG()
viz.add_nodes_from(used_nodes)
viz.add_edges_from([e for e, _ in stable_edges])

g = viz.to_graphviz()

for e in g.edges():
    u, v = e
    f = edge_freq[(u, v)]
    e.attr.update({
        "label": f" {f:.2f}",
        "penwidth": str(0.1 + 1.0 * f),
        "arrowsize": "0.4",
        "color": "#404040",
        "fontsize": "5", "fontcolor":"#404040", "fontname": "Helvetica"
    })

g.graph_attr.update({
    "rankdir": "LR", "dpi": "1200", "size": 5,
    "splines": "polyline", "overlap": "false", "bgcolor": "white",
})

g.layout(prog="dot")
# display(g)
g.draw("bootstrapped_survey_structure.jpg")

# Cleanliness & Comfort satisfaction

In [7]:
cleanliness_columns = ['clean_score_routine', 'clean_hours_since_routine', 'clean_score_deep', 'clean_days_since_deep', 'last_clean_score', 'hours_since_last_clean', 'clean_has_prior_routine', 'clean_has_prior_deep', 'lda_hours_since', 'Satisfaction', 'Cleanliness_Sat', 'Comfort_Sat', 'Recommendation']
cleanliness_df = main_block[cleanliness_columns]
print(len(cleanliness_df))
cleanliness_df.isna().sum()

27177


clean_score_routine          13015
clean_hours_since_routine    13015
clean_score_deep             15517
clean_days_since_deep        15517
last_clean_score             13015
hours_since_last_clean       13015
clean_has_prior_routine          0
clean_has_prior_deep             0
lda_hours_since                  0
Satisfaction                     0
Cleanliness_Sat                  0
Comfort_Sat                      0
Recommendation                   0
dtype: int64

In [8]:
cleanliness_df.dropna(subset=['clean_score_routine', 'clean_score_deep'], inplace=True)
cleanliness_df.drop(columns=['last_clean_score', 'hours_since_last_clean', 'clean_has_prior_routine', 'clean_has_prior_deep'], inplace=True)

C:\Users\Vincent\AppData\Local\Temp\ipykernel_7708\3617722352.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleanliness_df.dropna(subset=['clean_score_routine', 'clean_score_deep'], inplace=True)
C:\Users\Vincent\AppData\Local\Temp\ipykernel_7708\3617722352.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleanliness_df.drop(columns=['last_clean_score', 'hours_since_last_clean', 'clean_has_prior_routine', 'clean_has_prior_deep'], inplace=True)


In [ ]:
aspect_cols = ['Staff_Sat', 'Punctuality_Sat', 'Booking_Sat', 'Info_Sat', 'Departure_Sat',
               'Comfort_Sat', 'Cleanliness_Sat', 'Price_Sat', 'WiFi_Sat']
sinks   = ['Satisfaction', 'Recommendation']
survey  = aspect_cols + sinks
cleanliness_cols = [c for c in cleanliness_df.columns if c not in survey]

temporal_order_cleanliness = [cleanliness_cols, ['Comfort_Sat', 'Cleanliness_Sat'], ['Satisfaction'], ['Recommendation']]
required_edges = [("Satisfaction", "Recommendation")]

forbidden_cleanliness = [(a, b) for a in cleanliness_cols for b in cleanliness_cols if a != b]

expert_knowledge_cleanliness = ExpertKnowledge(temporal_order=temporal_order_cleanliness,
                                   required_edges=required_edges,
                                   forbidden_edges=forbidden_cleanliness)

print(f"Tiers:     {[len(t) for t in temporal_order_cleanliness]}")
print(f"Required:  {len(required_edges)}")
print(f"Forbidden: {len(forbidden_cleanliness)}")

Tiers:     [5, 2, 1, 1]
Required:  1
Forbidden: 20


In [11]:
hc_cleanliness = HillClimbSearch(scoring_method='bic-g', max_indegree=5, return_type='dag', expert_knowledge=expert_knowledge_cleanliness)
hc_cleanliness.fit(cleanliness_df)

cleanliness_structure = hc_cleanliness.causal_graph_
cleanliness_graph = cleanliness_structure.to_graphviz()
cleanliness_graph.graph_attr.update(rankdir='TB', dpi='1200')
cleanliness_graph.layout(prog='dot')
#display(cleanliness_graph)
cleanliness_graph.draw('initial_cleanliness_structure.jpg')

  0%|          | 6/1000000 [00:00<25:10:54, 11.03it/s]


In [ ]:
B = 1000
THRESHOLD = 0.5 
rng = np.random.default_rng(42)

edge_counts = Counter()
n_ok = 0

for b in tqdm(range(B)):
    sample = cleanliness_df.sample(n=len(cleanliness_df), replace=True, random_state=int(rng.integers(1e9)))
    # safety: a resample of a sparse column can go constant
    consts = [c for c in sample.columns if sample[c].nunique() <= 1]
    try:
        hc_cleanliness = HillClimbSearch(scoring_method="bic-g", max_indegree=5, return_type="dag",
                             expert_knowledge=expert_knowledge_cleanliness, show_progress=False)
        hc_cleanliness.fit(sample.drop(columns=consts) if consts else sample)
        edge_counts.update(list(hc_cleanliness.causal_graph_.edges()))
        n_ok += 1
    except Exception as e:
        print(f"bootstrap {b} skipped: {type(e).__name__}: {e}")

# edges that appear in > 50% of structures
edge_freq = {e: c / n_ok for e, c in edge_counts.items()}
stable_edges = sorted([(e, f) for e, f in edge_freq.items() if f > THRESHOLD],
                      key=lambda x: -x[1])

print(f"\n{n_ok}/{B} fits OK | {len(stable_edges)} stable edges (>{THRESHOLD:.0%}):\n")
for (u, v), f in stable_edges:
    print(f"  {f:5.0%}  {u} -> {v}")

100%|██████████| 1000/1000 [12:00<00:00,  1.39it/s]


1000/1000 fits OK | 7 stable edges (>50%):

   100%  Satisfaction -> Recommendation
   100%  Cleanliness_Sat -> Satisfaction
   100%  Comfort_Sat -> Satisfaction
   100%  Comfort_Sat -> Recommendation
   100%  Cleanliness_Sat -> Recommendation
    67%  Cleanliness_Sat -> Comfort_Sat
    50%  clean_days_since_deep -> Cleanliness_Sat


In [ ]:
used_nodes = {n for e, _ in stable_edges for n in e}
viz = DAG()
viz.add_nodes_from(used_nodes)
viz.add_edges_from([e for e, _ in stable_edges])

g = viz.to_graphviz()

for e in g.edges():
    u, v = e
    f = edge_freq[(u, v)]
    e.attr.update({
        "label": f" {f:.2f}",
        "penwidth": str(0.1 + 1.0 * f),
        "arrowsize": "0.4",
        "color": "#404040",
        "fontsize": "5", "fontcolor":"#404040", "fontname": "Helvetica"
    })

g.graph_attr.update({
    "rankdir": "LR", "dpi": "1200", "size": 5,
    "splines": "polyline", "overlap": "false", "bgcolor": "white",
})

g.layout(prog="dot")
# display(g)
g.draw("bootstrapped_cleanliness_and_comfort_structure.jpg")